# 01-1. ベクトル — 動かして確かめる

📖 解説: [`../01_vectors.md`](../01_vectors.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. ベクトルの作成と shape の確認
2. 加算・スカラー倍・引き算
3. 内積
4. ノルム (L¹, L², L∞)
5. コサイン類似度（推薦システムの基本）
6. 線形予測（ニューラルネット1ニューロン）
7. 高次元ベクトルの可視化

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (01_vectors.md)](../01_vectors.md)

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib inline

## 1. ベクトルを作る — NumPy と JAX

In [ ]:
# === 標準形式 ===
v = np.array([1.0, 2.0, 3.0])
print(f'値: {v}')
print(f'shape: {v.shape}    ← (3,) は1次元配列、3要素')
print(f'ndim: {v.ndim}      ← 1次元')
print(f'dtype: {v.dtype}')

In [ ]:
# === JAX形式 ===
v_jax = jnp.array([1.0, 2.0, 3.0])
print(f'値: {v_jax}')
print(f'shape: {v_jax.shape}')
print(f'dtype: {v_jax.dtype}    ← float32がデフォルト (NumPyはfloat64)')

## 2. 加算・スカラー倍・引き算

「2つの矢印を継ぎ足す」操作を可視化します。

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

print(f'u + v = {u + v}')
print(f'u - v = {u - v}')
print(f'3 * u = {3 * u}')
print(f'-1 * u = {-1 * u}    ← 反対向き')

In [ ]:
def plot_vector_addition(u_x: float, u_y: float, v_x: float, v_y: float) -> None:
    """2D ベクトル加算を可視化."""
    u = np.array([u_x, u_y])
    v = np.array([v_x, v_y])
    sum_uv = u + v

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, color='blue', label=f'u={tuple(u)}')
    ax.quiver(u[0], u[1], v[0], v[1], angles='xy', scale_units='xy', scale=1, color='green', label=f'v={tuple(v)} (uの先から)')
    ax.quiver(0, 0, sum_uv[0], sum_uv[1], angles='xy', scale_units='xy', scale=1, color='red', label=f'u+v={tuple(sum_uv)}')

    ax.set_xlim(-1, max(8, sum_uv[0] + 1))
    ax.set_ylim(-1, max(8, sum_uv[1] + 1))
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.set_title('ベクトルの加算: u + v')
    plt.show()

interact(
    plot_vector_addition,
    u_x=FloatSlider(min=-5, max=5, step=0.5, value=2),
    u_y=FloatSlider(min=-5, max=5, step=0.5, value=1),
    v_x=FloatSlider(min=-5, max=5, step=0.5, value=1),
    v_y=FloatSlider(min=-5, max=5, step=0.5, value=3),
);

## 3. 内積 — 「似ている度」の正体

内積は4通りの書き方ができますが、結果はすべて同じ。

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

print(f'u @ v       = {u @ v}        ← 推奨')
print(f'np.dot(u,v) = {np.dot(u, v)}')
print(f'np.inner    = {np.inner(u, v)}')
print(f'np.sum(u*v) = {np.sum(u * v)}    ← 定義どおり Σ uᵢvᵢ')

# JAX形式
print(f'JAX: {jnp.array(u) @ jnp.array(v)}')

## 4. ノルム — 「長さ」の3種類

「長さ」と一口に言っても、3種類あります。

In [ ]:
v = np.array([3.0, -4.0, 5.0])

print(f'L²ノルム (普通の長さ):    {np.linalg.norm(v, ord=2):.4f}')
print(f'L¹ノルム (マンハッタン):  {np.linalg.norm(v, ord=1):.4f}')
print(f'L∞ノルム (最大成分):      {np.linalg.norm(v, ord=np.inf):.4f}')

print()
print('検算:')
print(f'  L² = √(3² + 4² + 5²) = √{9+16+25} = {np.sqrt(50):.4f}')
print(f'  L¹ = |3| + |-4| + |5| = {3+4+5}')
print(f'  L∞ = max(|3|, |-4|, |5|) = {max(3, 4, 5)}')

In [ ]:
# 単位ベクトル化 (正規化)
v = np.array([3.0, 4.0])
v_unit = v / np.linalg.norm(v)
print(f'元のベクトル: {v},  ノルム: {np.linalg.norm(v)}')
print(f'単位ベクトル: {v_unit},  ノルム: {np.linalg.norm(v_unit):.4f}')
print('→ 向きはそのまま、長さだけ1になった')

## 5. コサイン類似度 — Netflix のおすすめロジック

5本の映画への評価ベクトルから、ユーザー間の「似ている度」を測ります。

In [ ]:
def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """コサイン類似度: -1 〜 +1."""
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

# 5本の映画への評価 (5段階)
alice = np.array([5, 4, 1, 5, 2])
bob   = np.array([4, 5, 2, 5, 1])
carol = np.array([1, 1, 5, 1, 5])

print(f'Alice vs Bob:   {cosine_similarity(alice, bob):.4f}    ← 似てる')
print(f'Alice vs Carol: {cosine_similarity(alice, carol):.4f}    ← 似てない')
print(f'Bob vs Carol:   {cosine_similarity(bob, carol):.4f}    ← 似てない')
print()
print('→ Alice にはBob のおすすめ映画を提案するのが合理的')

## 6. 直交性

In [ ]:
e1 = np.array([1, 0, 0])
e2 = np.array([0, 1, 0])
e3 = np.array([0, 0, 1])

print(f'e1 · e2 = {e1 @ e2}    ← 直交')
print(f'e1 · e3 = {e1 @ e3}    ← 直交')
print(f'e2 · e3 = {e2 @ e3}    ← 直交')

# コサイン類似度では 0
print(f'\ncos(e1, e2) = {cosine_similarity(e1, e2)}    ← 0 = 90度')

## 7. 線形予測 — ニューラルネットの1ニューロン

ニューラルネットの基本動作: `ŷ = w · x + b`

In [ ]:
def linear_predict(w: np.ndarray, x: np.ndarray, b: float) -> float:
    """線形モデル ŷ = w·x + b."""
    return float(w @ x + b)

# 例: 家賃を予測 (特徴: [広さm², 駅徒歩分, 築年数])
w = np.array([1500.0, -200.0, -800.0])  # 学習で決まる重み
b = 30000.0                              # バイアス

house = np.array([40.0, 5.0, 10.0])      # 40m², 駅徒歩5分, 築10年
rent = linear_predict(w, house, b)
print(f'予測家賃: {rent:,.0f} 円/月')

# 別の物件
house2 = np.array([60.0, 15.0, 25.0])    # 60m², 駅徒歩15分, 築25年
print(f'別の物件: {linear_predict(w, house2, b):,.0f} 円/月')

## 8. 高次元ベクトルの世界

現実の機械学習では、何百〜何千次元のベクトルを扱います。
OpenAI の text-embedding-3-small は 1536次元、large は 3072次元。

In [ ]:
# 1536次元のランダムなベクトルを2つ作る
rng = np.random.default_rng(seed=42)
embed_a = rng.standard_normal(1536)
embed_b = rng.standard_normal(1536)

print(f'shape: {embed_a.shape}')
print(f'A のノルム: {np.linalg.norm(embed_a):.4f}')
print(f'B のノルム: {np.linalg.norm(embed_b):.4f}')
print(f'コサイン類似度: {cosine_similarity(embed_a, embed_b):.4f}    ← ランダムなのでほぼ0 (直交)')

# B を A に少しだけ寄せると...
embed_c = 0.3 * embed_a + 0.7 * embed_b
print(f'\nA に少し寄せた C との類似度: {cosine_similarity(embed_a, embed_c):.4f}')

## まとめ

ここで触ったもの:
- ベクトルの作成・shape 確認
- 加算・スカラー倍・引き算
- 内積 (`@` 演算子)
- ノルム (L², L¹, L∞)
- コサイン類似度 (推薦システム)
- 線形予測 (ニューラルネット1ニューロン)
- 高次元ベクトル (LLM の Embedding)

## 次へ

次のノート → [`02_matrices.ipynb`](02_matrices.ipynb)

解説 → [`../02_matrices.md`](../02_matrices.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [章 TOP](../README.md) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`02_matrices.ipynb`](02_matrices.ipynb) |